In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
df = pd.read_csv("C:/Users/gdani/OneDrive/Desktop/master thesis/datasets/default credit card/default_of_credit_card_clients.csv", delimiter=";")
df

,Unnamed: 0,X1,X2,X3,X4,X5,X6,X7,X8,X9,...,X15,X16,X17,X18,X19,X20,X21,X22,X23,Y
0,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
1,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
2,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
3,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
4,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29996,29996,220000,1,3,1,39,0,0,0,0,...,88004,31237,15980,8500,20000,5003,3047,5000,1000,0
29997,29997,150000,1,3,2,43,-1,-1,-1,-1,...,8979,5190,0,1837,3526,8998,129,0,0,0
29998,29998,30000,1,2,2,37,4,3,2,-1,...,20878,20582,19357,0,0,22000,4200,2000,3100,1
29999,29999,80000,1,3,1,41,1,-1,0,0,...,52774,11855,48944,85900,3409,1178,1926,52964,1804,1


In [3]:
#let's start the data analysis
# Identify non-numeric columns
non_numeric = df.select_dtypes(exclude=['number'])

# Count them
num_non_numeric = non_numeric.shape[1]

print(f"Number of non-numeric variables: {num_non_numeric}")

Number of non-numeric variables: 25


In [4]:
#Feature selection

#Convert non numeric values in numeric values
df= df.apply(pd.to_numeric, errors='coerce')

#Check datatypes
print("\n",df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30001 entries, 0 to 30000
Data columns (total 25 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  30000 non-null  float64
 1   X1          30000 non-null  float64
 2   X2          30000 non-null  float64
 3   X3          30000 non-null  float64
 4   X4          30000 non-null  float64
 5   X5          30000 non-null  float64
 6   X6          30000 non-null  float64
 7   X7          30000 non-null  float64
 8   X8          30000 non-null  float64
 9   X9          30000 non-null  float64
 10  X10         30000 non-null  float64
 11  X11         30000 non-null  float64
 12  X12         30000 non-null  float64
 13  X13         30000 non-null  float64
 14  X14         30000 non-null  float64
 15  X15         30000 non-null  float64
 16  X16         30000 non-null  float64
 17  X17         30000 non-null  float64
 18  X18         30000 non-null  float64
 19  X19         30000 non-nul

In [5]:
# Identify non-numeric columns
non_numeric = df.select_dtypes(exclude=['number'])

# Count them
num_non_numeric = non_numeric.shape[1]

print(f"Number of non-numeric variables: {num_non_numeric}")

Number of non-numeric variables: 0


In [6]:
#drop missing values
df = df.dropna()
#Check
print("\n",df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 30000 entries, 1 to 30000
Data columns (total 25 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  30000 non-null  float64
 1   X1          30000 non-null  float64
 2   X2          30000 non-null  float64
 3   X3          30000 non-null  float64
 4   X4          30000 non-null  float64
 5   X5          30000 non-null  float64
 6   X6          30000 non-null  float64
 7   X7          30000 non-null  float64
 8   X8          30000 non-null  float64
 9   X9          30000 non-null  float64
 10  X10         30000 non-null  float64
 11  X11         30000 non-null  float64
 12  X12         30000 non-null  float64
 13  X13         30000 non-null  float64
 14  X14         30000 non-null  float64
 15  X15         30000 non-null  float64
 16  X16         30000 non-null  float64
 17  X17         30000 non-null  float64
 18  X18         30000 non-null  float64
 19  X19         30000 non-null  fl

In [24]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

from aif360.datasets import BinaryLabelDataset
from aif360.metrics import ClassificationMetric
from aif360.algorithms.postprocessing import CalibratedEqOddsPostprocessing

# Define label and protected attribute
label_col = 'Y'
protected_attr = 'X2'  # 1 = male (privileged), 2 = female (unprivileged)

# Split into features and target
X = df.drop(columns=[label_col])
y = df[label_col]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train base classifier
clf = LogisticRegression()
clf.fit(X_train_scaled, y_train)
y_pred = clf.predict(X_test_scaled)

# Prepare AIF360 datasets
aif_test = BinaryLabelDataset(
    favorable_label=1,
    unfavorable_label=0,
    #
    df=X_test.assign(Y=y_test.values),
    label_names=[label_col],
    protected_attribute_names=[protected_attr]
)

aif_pred = aif_test.copy()
aif_pred.labels = y_pred.reshape(-1, 1)

# Apply Calibrated Equalized Odds Postprocessing
eqodds = CalibratedEqOddsPostprocessing(
    privileged_groups=[{protected_attr: 1}],
    unprivileged_groups=[{protected_attr: 2}],
    cost_constraint="weighted"
)
eqodds = eqodds.fit(aif_test, aif_pred)
aif_eqodds_pred = eqodds.predict(aif_pred)

# Evaluate before fairness
metric_orig = ClassificationMetric(
    aif_test, aif_pred,
    privileged_groups=[{protected_attr: 1}],
    unprivileged_groups=[{protected_attr: 2}]
)

# Evaluate after fairness
metric_post = ClassificationMetric(
    aif_test, aif_eqodds_pred,
    privileged_groups=[{protected_attr: 1,}],
    unprivileged_groups=[{protected_attr: 2,}]
)

# Print results
print("=== BEFORE Fairness Postprocessing ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Equal Opportunity Difference:", metric_orig.equal_opportunity_difference())
print("Equalized Odds Difference:", metric_orig.average_odds_difference())
print("Disparate impact:", metric_orig.disparate_impact())

print("\n=== AFTER Calibrated Equalized Odds ===")
print("Accuracy:", accuracy_score(y_test, aif_eqodds_pred.labels))
print("Equal Opportunity Difference:", metric_post.equal_opportunity_difference())
print("Equalized Odds Difference:", metric_post.average_odds_difference())
print("Disparate impact:", metric_post.disparate_impact())

=== BEFORE Fairness Postprocessing ===
Accuracy: 0.8098333333333333
Equal Opportunity Difference: -0.01933693796243599
Equalized Odds Difference: -0.016389909366414945
Disparate impact: 0.772938794931972

=== AFTER Calibrated Equalized Odds ===
Accuracy: 1.0
Equal Opportunity Difference: 0.0
Equalized Odds Difference: 0.0
Disparate impact: 0.9001699260672549


In [25]:
from sklearn.metrics import confusion_matrix

# BEFORE fairness
y_true_before = aif_test.labels.ravel()
y_pred_before = aif_pred.labels.ravel()

# AFTER fairness
y_pred_after = aif_eqodds_pred.labels.ravel()

# Confusion matrices
cm_before = confusion_matrix(y_true_before, y_pred_before)
cm_after = confusion_matrix(y_true_before, y_pred_after)

# Unpack counts (assuming binary classification with labels 0 and 1)
tn_b, fp_b, fn_b, tp_b = cm_before.ravel()
tn_a, fp_a, fn_a, tp_a = cm_after.ravel()

# Print BEFORE fairness
print("=== BEFORE Fairness ===")
print(f"True Positives:  {tp_b}")
print(f"False Positives: {fp_b}")
print(f"True Negatives:  {tn_b}")
print(f"False Negatives: {fn_b}")

# Print AFTER fairness
print("\n=== AFTER Fairness ===")
print(f"True Positives:  {tp_a}")
print(f"False Positives: {fp_a}")
print(f"True Negatives:  {tn_a}")
print(f"False Negatives: {fn_a}")


=== BEFORE Fairness ===
True Positives:  309
False Positives: 137
True Negatives:  4550
False Negatives: 1004

=== AFTER Fairness ===
True Positives:  1313
False Positives: 0
True Negatives:  4687
False Negatives: 0
